# Solar Flare Prediction with ConvLSTM

This notebook trains a ConvLSTM model to predict solar wind flux from winding flux data cubes.

**Estimated training time:** ~30-60 minutes on Colab T4 GPU

## Setup Instructions

1. Upload your `.npy` data files to Google Drive
2. Update the `DATA_DIR` path in the Configuration cell
3. Run all cells in order

---


## 1. Setup & Dependencies


In [ ]:
# Install dependencies (most are pre-installed in Colab)
%pip install pyyaml tqdm -q

# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


## 2. Configuration

**Important:** Update `DATA_DIR` to point to your data folder in Google Drive.


In [ ]:
# =============================================================================
# CONFIGURATION - Modify these settings as needed
# =============================================================================

CONFIG = {
    # Device settings
    'use_cuda': True,  # Set to False to force CPU
    
    # Data settings
    'data_dir': '/content/drive/MyDrive/SolarFlare/data',  # Raw data path
    'use_preprocessed': False,  # Set True after running preprocessing cell
    'preprocessed_dir': '/content/data_processed',  # Preprocessed cubes path
    't_in': 8,           # Input sequence length
    't_out': 3,          # Output sequence length
    'train_split': 0.7,
    'val_split': 0.15,
    'augment': True,
    
    # Normalization
    'norm_method': 'robust',  # 'robust' or 'fixed'
    'percentile_low': 1,
    'percentile_high': 99,
    'fixed_factor': 40000.0,
    
    # Model architecture
    'input_channels': 1,
    'channels': [16, 32, 64],  # Channel progression
    'kernel_size': 3,
    'downsample_input': True,  # 2x spatial reduction (saves memory)
    
    # Training
    'batch_size': 1,
    'epochs': 25,        # ~1 hour total
    'lr': 0.001,
    'weight_decay': 1e-5,
    'tf_start': 0.5,     # Teacher forcing start ratio
    'patience': 8,       # Early stopping patience
    'use_amp': True,     # Mixed precision training
    'grad_clip': 1.0,
    
    # Output
    'save_dir': '/content/outputs',
}

print("Configuration loaded!")
print(f"Data directory: {CONFIG['data_dir']}")


## 2.1 Optional: Preprocess Data (Run Once)

Run this cell to convert raw .npy files to dense cubes. This speeds up subsequent data loading significantly. After running, set `CONFIG['use_preprocessed'] = True` above.


In [ ]:
# OPTIONAL: Run this cell once to preprocess data for faster loading
# After running, set CONFIG['use_preprocessed'] = True in the config cell above

import os
from tqdm.auto import tqdm

def preprocess_data(input_dir, output_dir, norm_method='robust'):
    """Convert raw .npy files to preprocessed cubes."""
    from pathlib import Path
    import json
    
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    npy_files = sorted(input_path.glob('windTotal*.npy'))
    if len(npy_files) == 0:
        npy_files = sorted(input_path.glob('*.npy'))
    
    if len(npy_files) == 0:
        raise FileNotFoundError(f"No .npy files found in {input_path}")
    
    print(f"Found {len(npy_files)} raw data files")
    
    cubes = []
    cube_metadata = []
    
    for file_path in npy_files:
        print(f"\nProcessing {file_path.name}...")
        data = np.load(file_path)
        
        x_coords = np.unique(data['X'])
        y_coords = np.unique(data['Y'])
        times = np.unique(data['time'])
        H, W, T = len(y_coords), len(x_coords), len(times)
        
        x_to_idx = {x: i for i, x in enumerate(x_coords)}
        y_to_idx = {y: i for i, y in enumerate(y_coords)}
        time_to_idx = {t: i for i, t in enumerate(times)}
        
        flux_cube = np.zeros((T, H, W), dtype=np.float32)
        print(f"  Converting {len(data):,} records to cube ({T}, {H}, {W})...")
        
        for i in tqdm(range(len(data)), desc="  Building cube"):
            t_idx = time_to_idx[data['time'][i]]
            h_idx = y_to_idx[data['Y'][i]]
            w_idx = x_to_idx[data['X'][i]]
            flux_cube[t_idx, h_idx, w_idx] = data['windTotal'][i]
        
        cubes.append(flux_cube)
        cube_metadata.append({'source_file': file_path.name, 'shape': (T, H, W)})
    
    # Compute normalization
    all_values = np.concatenate([c.flatten()[::100] for c in cubes])
    if norm_method == 'robust':
        center = float(np.median(all_values))
        scale = float(max((np.percentile(all_values, 99) - np.percentile(all_values, 1)) / 2, 1.0))
    else:
        center, scale = 0.0, 40000.0
    
    print(f"\nNormalization: center={center:.2f}, scale={scale:.2f}")
    
    # Save cubes
    for i, (cube, meta) in enumerate(zip(cubes, cube_metadata)):
        cube_norm = (cube - center) / scale
        output_file = output_path / f"cube_{i:03d}.npz"
        np.savez_compressed(output_file, data=cube_norm, **meta)
        print(f"  Saved {output_file.name}")
    
    # Save metadata
    metadata = {'normalization': {'center': center, 'scale': scale}, 'cubes': cube_metadata}
    with open(output_path / 'metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2, default=str)
    
    print(f"\nPreprocessing complete! {len(cubes)} cubes saved to {output_path}")

# Uncomment to run preprocessing:
# preprocess_data(CONFIG['data_dir'], CONFIG['preprocessed_dir'])


## 3. Model Architecture

### ConvLSTM Components


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import List, Tuple, Optional


class ConvLSTMCell(nn.Module):
    """Single ConvLSTM cell - combines CNN spatial processing with LSTM memory."""
    
    def __init__(self, input_dim: int, hidden_dim: int, kernel_size: int, bias: bool = True):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.padding = kernel_size // 2
        
        self.conv = nn.Conv2d(
            in_channels=input_dim + hidden_dim,
            out_channels=4 * hidden_dim,
            kernel_size=kernel_size,
            padding=self.padding,
            bias=bias
        )
        self._init_forget_bias()
    
    def _init_forget_bias(self):
        with torch.no_grad():
            if self.conv.bias is not None:
                self.conv.bias[self.hidden_dim:2 * self.hidden_dim].fill_(1.0)
    
    def forward(self, x, h_prev, c_prev):
        combined = torch.cat([x, h_prev], dim=1)
        gates = self.conv(combined)
        i, f, g, o = torch.split(gates, self.hidden_dim, dim=1)
        
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        g = torch.tanh(g)
        o = torch.sigmoid(o)
        
        c_next = f * c_prev + i * g
        h_next = o * torch.tanh(c_next)
        
        return h_next, c_next


class ConvLSTM(nn.Module):
    """Multi-layer ConvLSTM that processes a sequence of spatial inputs."""
    
    def __init__(self, input_dim: int, hidden_dim: int, kernel_size: int = 3, num_layers: int = 1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        cells = []
        for i in range(num_layers):
            cur_input = input_dim if i == 0 else hidden_dim
            cells.append(ConvLSTMCell(cur_input, hidden_dim, kernel_size))
        self.cell_list = nn.ModuleList(cells)
    
    def forward(self, x, hidden_state=None):
        B, _, T, H, W = x.size()
        
        if hidden_state is None:
            hidden_state = self._init_hidden(B, H, W, x.device)
        
        outputs = []
        for t in range(T):
            x_t = x[:, :, t]
            for layer_idx, cell in enumerate(self.cell_list):
                h_prev, c_prev = hidden_state[layer_idx]
                h_next, c_next = cell(x_t, h_prev, c_prev)
                hidden_state[layer_idx] = (h_next, c_next)
                x_t = h_next
            outputs.append(h_next)
        
        return torch.stack(outputs, dim=2), hidden_state
    
    def _init_hidden(self, batch_size, height, width, device):
        return [
            (torch.zeros(batch_size, self.hidden_dim, height, width, device=device),
             torch.zeros(batch_size, self.hidden_dim, height, width, device=device))
            for _ in range(self.num_layers)
        ]


print("ConvLSTM components defined!")


In [ ]:
class SolarFluxPredictor(nn.Module):
    """ConvLSTM-based autoregressive solar flux predictor."""
    
    def __init__(self, input_channels=1, t_out=3, channels=[16, 32, 64], 
                 kernel_size=3, downsample_input=True):
        super().__init__()
        self.t_out = t_out
        self.downsample_input = downsample_input
        
        c1, c2, c3 = channels
        
        # Optional input downsampling
        if downsample_input:
            self.input_down = nn.Sequential(
                nn.Conv2d(input_channels, c1, kernel_size=4, stride=2, padding=1),
                nn.ReLU(inplace=True)
            )
        else:
            self.input_down = None
        
        # Preprocessing
        self.preprocess = nn.Sequential(
            nn.Conv2d(c1 if downsample_input else input_channels, c1, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        
        # Encoder
        self.encoder_conv1 = ConvLSTM(c1, c1, kernel_size)
        self.downsample1 = nn.Conv2d(c1, c2, kernel_size=3, stride=2, padding=1)
        self.encoder_conv2 = ConvLSTM(c2, c2, kernel_size)
        self.encoder_conv3 = ConvLSTM(c2, c3, kernel_size)
        
        # Decoder
        self.decoder_input_conv = nn.Conv2d(input_channels, c1, kernel_size=3, padding=1)
        self.decoder_proj = nn.Conv2d(c1, c2, kernel_size=3, stride=2, padding=1)
        self.decoder_conv2 = ConvLSTM(c2, c2, kernel_size)
        self.decoder_conv3 = ConvLSTM(c2, c3, kernel_size)
        
        self.upsample = nn.ConvTranspose2d(c3, c2, kernel_size=4, stride=2, padding=1)
        self.refine_conv = ConvLSTM(c2 + c1, c1, kernel_size)
        
        # Output head
        if downsample_input:
            self.output_conv = nn.Sequential(
                nn.ConvTranspose2d(c1, c1, kernel_size=4, stride=2, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(c1, input_channels, kernel_size=1)
            )
        else:
            self.output_conv = nn.Conv2d(c1, input_channels, kernel_size=1)
    
    def forward(self, x, teacher_forcing_ratio=0.0, y_true=None):
        B, C, T_in, H_orig, W_orig = x.size()
        
        # Optional input downsampling
        if self.downsample_input:
            x_flat = x.view(B * T_in, C, H_orig, W_orig)
            x_down = self.input_down(x_flat)
            _, c1, H_down, W_down = x_down.shape
            x_prep = x_down.view(B, c1, T_in, H_down, W_down)
        else:
            x_flat = x.view(B * T_in, C, H_orig, W_orig)
            x_prep = self.preprocess(x_flat)
            _, c1, H_down, W_down = x_prep.shape
            x_prep = x_prep.view(B, c1, T_in, H_down, W_down)
        
        # Preprocess
        x_prep_flat = x_prep.view(B * T_in, -1, H_down, W_down)
        x_prep_out = self.preprocess(x_prep_flat)
        x_prep = x_prep_out.view(B, -1, T_in, H_down, W_down)
        
        # Encoder
        h1_seq, h1_states = self.encoder_conv1(x_prep)
        h1_skip = h1_states[0][0]
        
        h1_down = self.downsample1(h1_seq[:, :, -1])
        h1_down = h1_down.unsqueeze(2).expand(-1, -1, T_in, -1, -1)
        
        h2_seq, h2_states = self.encoder_conv2(h1_down)
        h3_seq, h3_states = self.encoder_conv3(h2_seq)
        
        # Decoder (autoregressive)
        predictions = []
        input_frame = x[:, :, -1]
        
        decoder_state2 = [(h2_states[0][0].clone(), h2_states[0][1].clone())]
        decoder_state3 = [(h3_states[0][0].clone(), h3_states[0][1].clone())]
        refine_state = None
        
        for t in range(self.t_out):
            if self.downsample_input:
                dec_input = self.input_down(input_frame)
            else:
                dec_input = self.decoder_input_conv(input_frame)
            
            dec_down = self.decoder_proj(dec_input).unsqueeze(2)
            dec_h2, decoder_state2 = self.decoder_conv2(dec_down, decoder_state2)
            dec_h3, decoder_state3 = self.decoder_conv3(dec_h2, decoder_state3)
            
            dec_up = self.upsample(dec_h3[:, :, 0])
            # Ensure dimensions match (handles odd spatial sizes)
            if dec_up.shape[2:] != h1_skip.shape[2:]:
                dec_up = F.interpolate(dec_up, size=h1_skip.shape[2:], mode='bilinear', align_corners=False)
            dec_concat = torch.cat([dec_up, h1_skip], dim=1).unsqueeze(2)
            refined, refine_state = self.refine_conv(dec_concat, refine_state)
            
            delta = self.output_conv(refined[:, :, 0])
            # Ensure delta matches original input dimensions
            if delta.shape[2:] != (H_orig, W_orig):
                delta = F.interpolate(delta, size=(H_orig, W_orig), mode='bilinear', align_corners=False)
            pred_frame = input_frame + delta
            predictions.append(pred_frame)
            
            # Teacher forcing
            if teacher_forcing_ratio > 0 and y_true is not None and np.random.rand() < teacher_forcing_ratio:
                input_frame = y_true[:, :, t]
            else:
                input_frame = pred_frame
        
        return torch.stack(predictions, dim=2)
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


print("SolarFluxPredictor defined!")


In [ ]:
from torch.utils.data import Dataset, DataLoader
from pathlib import Path


class SolarFluxDataset(Dataset):
    """Dataset for solar flux prediction with sliding windows."""
    
    def __init__(self, samples, datasets, t_in=8, t_out=3, augment=True):
        self.samples = samples
        self.datasets = datasets
        self.t_in = t_in
        self.t_out = t_out
        self.augment = augment
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        dataset_id, start_idx = self.samples[idx]
        data = self.datasets[dataset_id]
        
        X_in = data[start_idx:start_idx + self.t_in]
        Y_out = data[start_idx + self.t_in:start_idx + self.t_in + self.t_out]
        
        if self.augment:
            if np.random.rand() > 0.5:
                X_in = np.flip(X_in, axis=2).copy()
                Y_out = np.flip(Y_out, axis=2).copy()
            if np.random.rand() > 0.5:
                X_in = np.flip(X_in, axis=1).copy()
                Y_out = np.flip(Y_out, axis=1).copy()
        
        X_in = torch.from_numpy(X_in.copy()).float().unsqueeze(0)
        Y_out = torch.from_numpy(Y_out.copy()).float().unsqueeze(0)
        
        return X_in, Y_out, (dataset_id, start_idx)


def load_and_prepare_data(config):
    """Load .npy files and create train/val/test datasets."""
    data_path = Path(config['data_dir'])
    
    if not data_path.exists():
        raise FileNotFoundError(f"Data directory not found: {data_path}")
    
    npy_files = sorted(data_path.glob('windTotal*.npy'))
    if len(npy_files) == 0:
        npy_files = sorted(data_path.glob('*.npy'))
    
    if len(npy_files) == 0:
        raise FileNotFoundError(f"No .npy files found in {data_path}")
    
    print(f"Found {len(npy_files)} data files")
    
    datasets = []
    all_values = []
    
    for file_path in npy_files:
        print(f"Loading {file_path.name}...")
        data = np.load(file_path)
        
        x_coords = np.unique(data['X'])
        y_coords = np.unique(data['Y'])
        times = np.unique(data['time'])
        
        H, W, T = len(y_coords), len(x_coords), len(times)
        print(f"  Shape: T={T}, H={H}, W={W}")
        
        x_to_idx = {x: i for i, x in enumerate(x_coords)}
        y_to_idx = {y: i for i, y in enumerate(y_coords)}
        time_to_idx = {t: i for i, t in enumerate(times)}
        
        flux_cube = np.zeros((T, H, W), dtype=np.float32)
        for i in range(len(data)):
            t_idx = time_to_idx[data['time'][i]]
            h_idx = y_to_idx[data['Y'][i]]
            w_idx = x_to_idx[data['X'][i]]
            flux_cube[t_idx, h_idx, w_idx] = data['windTotal'][i]
        
        datasets.append(flux_cube)
        all_values.append(flux_cube.flatten()[::100])
    
    # Normalization
    all_values = np.concatenate(all_values)
    
    if config['norm_method'] == 'robust':
        low = np.percentile(all_values, config['percentile_low'])
        high = np.percentile(all_values, config['percentile_high'])
        center = np.median(all_values)
        scale = max((high - low) / 2, 1.0)
    else:
        center = 0.0
        scale = config['fixed_factor']
    
    print(f"\nNormalization: center={center:.2f}, scale={scale:.2f}")
    
    for i in range(len(datasets)):
        datasets[i] = (datasets[i] - center) / scale
    
    # Create samples
    t_in, t_out = config['t_in'], config['t_out']
    train_samples, val_samples, test_samples = [], [], []
    
    for ds_id, cube in enumerate(datasets):
        T = cube.shape[0]
        n_samples = T - t_in - t_out + 1
        if n_samples <= 0:
            continue
        
        train_end = int(config['train_split'] * n_samples)
        val_end = int((config['train_split'] + config['val_split']) * n_samples)
        
        for idx in range(n_samples):
            sample = (ds_id, idx)
            if idx < train_end:
                train_samples.append(sample)
            elif idx < val_end:
                val_samples.append(sample)
            else:
                test_samples.append(sample)
    
    print(f"\nSamples: Train={len(train_samples)}, Val={len(val_samples)}, Test={len(test_samples)}")
    
    train_ds = SolarFluxDataset(train_samples, datasets, t_in, t_out, augment=config['augment'])
    val_ds = SolarFluxDataset(val_samples, datasets, t_in, t_out, augment=False)
    test_ds = SolarFluxDataset(test_samples, datasets, t_in, t_out, augment=False)
    
    return train_ds, val_ds, test_ds


def load_preprocessed_data(config):
    """Load preprocessed cube files for fast training."""
    import json
    
    data_path = Path(config['preprocessed_dir'])
    
    if not data_path.exists():
        raise FileNotFoundError(f"Preprocessed directory not found: {data_path}. Run preprocessing first.")
    
    metadata_file = data_path / 'metadata.json'
    if not metadata_file.exists():
        raise FileNotFoundError(f"metadata.json not found. Run preprocessing first.")
    
    with open(metadata_file, 'r') as f:
        metadata = json.load(f)
    
    print(f"Loading preprocessed data from {data_path}")
    print(f"  Normalization: center={metadata['normalization']['center']:.2f}, scale={metadata['normalization']['scale']:.2f}")
    
    cube_files = sorted(data_path.glob('cube_*.npz'))
    datasets = []
    
    for cube_file in cube_files:
        print(f"  Loading {cube_file.name}...")
        npz = np.load(cube_file)
        datasets.append(npz['data'])
    
    # Create samples
    t_in, t_out = config['t_in'], config['t_out']
    train_samples, val_samples, test_samples = [], [], []
    
    for ds_id, cube in enumerate(datasets):
        T = cube.shape[0]
        n_samples = T - t_in - t_out + 1
        if n_samples <= 0:
            continue
        
        train_end = int(config['train_split'] * n_samples)
        val_end = int((config['train_split'] + config['val_split']) * n_samples)
        
        for idx in range(n_samples):
            sample = (ds_id, idx)
            if idx < train_end:
                train_samples.append(sample)
            elif idx < val_end:
                val_samples.append(sample)
            else:
                test_samples.append(sample)
    
    print(f"\nSamples: Train={len(train_samples)}, Val={len(val_samples)}, Test={len(test_samples)}")
    
    train_ds = SolarFluxDataset(train_samples, datasets, t_in, t_out, augment=config['augment'])
    val_ds = SolarFluxDataset(val_samples, datasets, t_in, t_out, augment=False)
    test_ds = SolarFluxDataset(test_samples, datasets, t_in, t_out, augment=False)
    
    return train_ds, val_ds, test_ds


print("Data loading functions defined!")


In [ ]:
import torch.nn.functional as F
from tqdm.auto import tqdm
from contextlib import nullcontext
import json


def get_device(use_cuda=True):
    """Get appropriate device."""
    if use_cuda and torch.cuda.is_available():
        device = torch.device('cuda')
        print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device('cpu')
        print("Using CPU")
    return device


def get_amp_context(use_amp, device):
    """Get autocast context for mixed precision."""
    if use_amp and device.type == 'cuda':
        return torch.amp.autocast(device_type='cuda')
    return nullcontext()


class DummyScaler:
    """No-op gradient scaler for CPU."""
    def scale(self, loss): return loss
    def step(self, opt): opt.step()
    def update(self): pass
    def unscale_(self, opt): pass


def get_grad_scaler(use_amp, device):
    """Get gradient scaler."""
    if use_amp and device.type == 'cuda':
        return torch.amp.GradScaler()
    return DummyScaler()


def train_epoch(model, loader, optimizer, scaler, device, tf_ratio, epoch, config):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    
    pbar = tqdm(loader, desc=f"Epoch {epoch}")
    for X_in, Y_out, _ in pbar:
        X_in, Y_out = X_in.to(device), Y_out.to(device)
        
        optimizer.zero_grad()
        
        with get_amp_context(config['use_amp'], device):
            predictions = model(X_in, tf_ratio, Y_out)
            loss = F.l1_loss(predictions, Y_out)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip'])
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.6f}'})
    
    return total_loss / len(loader)


def validate(model, loader, device, config):
    """Validate model."""
    model.eval()
    total_loss = 0.0
    all_mae = []
    
    with torch.no_grad():
        for X_in, Y_out, _ in tqdm(loader, desc="Validating"):
            X_in, Y_out = X_in.to(device), Y_out.to(device)
            
            with get_amp_context(config['use_amp'], device):
                predictions = model(X_in, 0.0)
                loss = F.l1_loss(predictions, Y_out)
            
            total_loss += loss.item()
            mae = F.l1_loss(predictions, Y_out, reduction='none').mean(dim=(0,1,3,4))
            all_mae.append(mae.cpu().numpy())
    
    return total_loss / len(loader), np.mean(all_mae, axis=0)


print("Training functions defined!")


## 6. Main Training Loop


In [ ]:
def train_model(model, train_loader, val_loader, config, device):
    """Main training loop with early stopping."""
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config['weight_decay']
    )
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config['epochs'], eta_min=1e-6
    )
    
    scaler = get_grad_scaler(config['use_amp'], device)
    
    best_val_loss = float('inf')
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': [], 'val_mae': []}
    
    save_path = Path(config['save_dir'])
    save_path.mkdir(parents=True, exist_ok=True)
    
    print(f"\nStarting training for {config['epochs']} epochs")
    print(f"AMP: {config['use_amp']}, Device: {device}\n")
    
    for epoch in range(1, config['epochs'] + 1):
        tf_ratio = max(0.0, config['tf_start'] * (1 - epoch / config['epochs']))
        
        print(f"Epoch {epoch}/{config['epochs']} | LR: {optimizer.param_groups[0]['lr']:.2e} | TF: {tf_ratio:.3f}")
        
        train_loss = train_epoch(model, train_loader, optimizer, scaler, device, tf_ratio, epoch, config)
        val_loss, val_mae = validate(model, val_loader, device, config)
        
        scheduler.step()
        
        print(f"  Train Loss: {train_loss:.6f}")
        print(f"  Val Loss:   {val_loss:.6f}")
        print(f"  Val MAE:    {val_mae}")
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_mae'].append(val_mae.tolist())
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_loss': val_loss,
            }, save_path / 'best_model.pt')
            print(f"  ✓ Saved best model")
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"  No improvement ({patience_counter}/{config['patience']})")
        
        if patience_counter >= config['patience']:
            print(f"\nEarly stopping at epoch {epoch}")
            break
        
        print()
    
    with open(save_path / 'history.json', 'w') as f:
        json.dump(history, f, indent=2)
    
    return history


print("Training loop defined!")


## 7. Visualization


In [ ]:
import matplotlib.pyplot as plt


def plot_history(history):
    """Plot training history."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].plot(history['train_loss'], label='Train')
    axes[0].plot(history['val_loss'], label='Val')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].set_title('Training Progress')
    axes[0].grid(True, alpha=0.3)
    
    mae_arr = np.array(history['val_mae'])
    for t in range(mae_arr.shape[1]):
        axes[1].plot(mae_arr[:, t], label=f't+{t+1}')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MAE')
    axes[1].legend()
    axes[1].set_title('Val MAE per Timestep')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{CONFIG['save_dir']}/training_history.png", dpi=150)
    plt.show()


def visualize_predictions(model, dataset, device, n_samples=3):
    """Visualize predictions vs ground truth."""
    model.eval()
    t_out = dataset.t_out
    
    fig, axes = plt.subplots(n_samples, 2 + t_out, figsize=(4 * (2 + t_out), 4 * n_samples))
    if n_samples == 1:
        axes = axes.reshape(1, -1)
    
    with torch.no_grad():
        for i in range(n_samples):
            idx = i * len(dataset) // n_samples
            X_in, Y_out, _ = dataset[idx]
            X_in = X_in.unsqueeze(0).to(device)
            Y_out = Y_out.unsqueeze(0).to(device)
            
            preds = model(X_in, 0.0)
            
            axes[i, 0].imshow(X_in[0, 0, -1].cpu(), cmap='RdBu_r', vmin=-1, vmax=1)
            axes[i, 0].set_title(f'Input t={dataset.t_in}')
            axes[i, 0].axis('off')
            
            for t in range(t_out):
                axes[i, 1+t].imshow(preds[0, 0, t].cpu(), cmap='RdBu_r', vmin=-1, vmax=1)
                axes[i, 1+t].set_title(f'Pred t+{t+1}')
                axes[i, 1+t].axis('off')
            
            axes[i, -1].imshow(Y_out[0, 0, -1].cpu(), cmap='RdBu_r', vmin=-1, vmax=1)
            axes[i, -1].set_title(f'GT t+{t_out}')
            axes[i, -1].axis('off')
    
    plt.tight_layout()
    plt.savefig(f"{CONFIG['save_dir']}/predictions.png", dpi=150)
    plt.show()


print("Visualization functions defined!")


---

## 8. Run Training

Execute these cells to start the training process.


In [ ]:
# =============================================================================
# LOAD DATA
# =============================================================================
print("=" * 60)
print("LOADING DATA")
print("=" * 60)

if CONFIG.get('use_preprocessed', False):
    # Fast loading from preprocessed cubes
    train_dataset, val_dataset, test_dataset = load_preprocessed_data(CONFIG)
else:
    # Load from raw structured arrays (slower)
    train_dataset, val_dataset, test_dataset = load_and_prepare_data(CONFIG)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)


In [ ]:
# =============================================================================
# CREATE MODEL
# =============================================================================
print("\n" + "=" * 60)
print("CREATING MODEL")
print("=" * 60)

device = get_device(CONFIG['use_cuda'])

model = SolarFluxPredictor(
    input_channels=CONFIG['input_channels'],
    t_out=CONFIG['t_out'],
    channels=CONFIG['channels'],
    kernel_size=CONFIG['kernel_size'],
    downsample_input=CONFIG['downsample_input']
).to(device)

print(f"Parameters: {model.count_parameters():,}")


In [ ]:
# =============================================================================
# TRAIN
# =============================================================================
print("\n" + "=" * 60)
print("TRAINING")
print("=" * 60)

history = train_model(model, train_loader, val_loader, CONFIG, device)


In [ ]:
# =============================================================================
# PLOT TRAINING HISTORY
# =============================================================================
plot_history(history)


In [ ]:
# =============================================================================
# TEST
# =============================================================================
print("\n" + "=" * 60)
print("TESTING")
print("=" * 60)

# Load best model
checkpoint = torch.load(f"{CONFIG['save_dir']}/best_model.pt")
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']}")

test_loss, test_mae = validate(model, test_loader, device, CONFIG)
print(f"\nTest Loss: {test_loss:.6f}")
print(f"Test MAE per timestep: {test_mae}")


In [ ]:
# =============================================================================
# VISUALIZE PREDICTIONS
# =============================================================================
print("\n" + "=" * 60)
print("VISUALIZING PREDICTIONS")
print("=" * 60)

visualize_predictions(model, test_dataset, device, n_samples=3)


## 9. Download Results

Run this cell to download the trained model and results.


In [ ]:
# Download results
from google.colab import files
import shutil

# Create zip of outputs
shutil.make_archive('/content/solarflare_results', 'zip', CONFIG['save_dir'])

# Download
files.download('/content/solarflare_results.zip')
print("\nDownload started! Check your browser downloads.")


---

## Notes

- **Training time:** ~30-60 minutes on T4 GPU
- **VRAM usage:** ~3-4 GB
- **Early stopping:** Will stop if validation loss doesn't improve for 8 epochs

### Troubleshooting

**Out of memory?**
- Reduce channels: `CONFIG['channels'] = [8, 16, 32]`
- Ensure `downsample_input = True`

**Session disconnecting?**
- Keep the browser tab active
- Training should complete within Colab's limits

**Data not found?**
- Check that `CONFIG['data_dir']` points to your Google Drive folder
- Ensure `.npy` files are uploaded to that folder
